### Dataset and Task Metadata

In [10]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="heart_disease_va_long_beach",
    dataset_year="1989",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C52P4X",
    download_description="""
Get the UCI data.

wget https://archive.ics.uci.edu/static/public/45/heart+disease.zip && unzip heart+disease.zip processed.va.data && rm heart+disease.zip && mkdir -p local-data-warehouse/heart_disease_va_long_beach && mv processed.va.data local-data-warehouse/heart_disease_va_long_beach/
""",
    # References
    academic_reference_bibtex="""@article{detrano1989international,
  title={International application of a new probability algorithm for the diagnosis of coronary artery disease},
  author={Detrano, Robert and Janosi, Andras and Steinbrunn, Walter and Pfisterer, Matthias and Schmid, Johann-Jakob and Sandhu, Sarbjit and Guppy, Kern H and Lee, Stella and Froelicher, Victor},
  journal={The American journal of cardiology},
  volume={64},
  number={5},
  pages={304--310},
  year={1989},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="detrano1989international",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the processed version and the subset of 14 attributes used in the study and clinical practice.

- We encode missing values as np.nan instead of "?".
- We make the target binary (0=no heart disease, 1=heart disease). This follows the original study in attempting to distinguish presence (values 1,2,3,4) from absence (value 0).
- The dataset has one naturally occurring duplicate, which we do not drop.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="heart_disease_diagnosis",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="heart_disease_diagnosis",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

columns = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "num"
]

df = pd.read_csv(dataset_mold.path / "processed.va.data", header=None, names=columns)
print("Loaded data shape:", df.shape)

df = df.replace("?", np.nan)
df[["ca", "trestbps", "thalach", "oldpeak", "chol"]] = df[["ca", "trestbps", "thalach", "oldpeak", "chol"]].astype(float)
# Make target
df["heart_disease_diagnosis"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])

as_cat_type = ["thal", "slope", "exang", "restecg", "fbs", "cp", "sex", "heart_disease_diagnosis"]
df[as_cat_type] = df[as_cat_type].astype("category")


df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (200, 14)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 200
Columns: 14
Use sampling: False (sample size: 200)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['chol', 'thalach', 'trestbps', 'age', 'oldpeak', 'cp', 'thal', 'restecg', 'slope', 'exang']
Rows remaining as candidates after top-10 filter: 2 (of 200)

#### Duplicate Report
Total duplicate rows: 1 (0.50% of dataset)
Duplicate rows ignoring target: 1 (0.50% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,heart_disease_diagnosis
0,63,1,4,160.0,267.0,1,1,88.0,1,2.0,NaN,NaN,NaN,1
1,62,1,4,120.0,220.0,0,1,86.0,0,0.0,NaN,NaN,NaN,0
2,54,1,4,NaN,0.0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,1
3,69,1,4,NaN,210.0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,1
4,61,0,2,140.0,298.0,1,0,120.0,1,0.0,NaN,NaN,7,0


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,thal,category,166.0,83.0,3.0,"7, 6, 3"
1,slope,category,102.0,51.0,3.0,"2, 3, 1"
2,exang,category,53.0,26.5,2.0,"1, 0"
3,fbs,category,7.0,3.5,2.0,"0, 1"
4,sex,category,0.0,0.0,2.0,"1, 0"
5,cp,category,0.0,0.0,4.0,"4, 3, 2, 1"
6,restecg,category,0.0,0.0,3.0,"1, 0, 2"
7,heart_disease_diagnosis,category,0.0,0.0,2.0,"1, 0"
8,ca,float64,198.0,99.0,1.0,0.0
9,trestbps,float64,56.0,28.0,40.0,"120.0, 130.0, 140.0, 110.0, 150.0, 160.0, 122.0, 142.0, 126.0, 136.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,200.0,59.350000,7.811697,35.0,77.0
trestbps,144.0,133.763889,21.537733,0.0,190.0
chol,193.0,178.746114,114.035232,0.0,458.0
thalach,147.0,122.795918,21.990328,69.0,180.0
oldpeak,144.0,1.320833,1.106236,-0.5,4.0
ca,2.0,0.000000,0.000000,0.0,0.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count   pct
column                  rank                   
cp                      1        4    131  65.5
                        2        3     47  23.5
                        3        2     14   7.0
                        4        1      8   4.0
exang                   1        1     95  47.5
                        2     <NA>     53  26.5
                        3        0     52  26.0
fbs                     1        0    125  62.5
                        2        1     68  34.0
                        3     <NA>      7   3.5
heart_disease_diagnosis 1        1    149  74.5
                        2        0     51  25.5
restecg                 1        1     93  46.5
                        2        0     80  40.0
                        3        2     27  13.5
sex                     1        1    194  97.0
                        2        0      6   3.0
slope                   1     <NA>    102  51.0
                        2        2     53  26.5
                        3        3     29  14.5
                        4        1     16   8.0
thal                    1     <NA>    166  83.0
                        2        7     22  11.0
                        3        6      8   4.0
                        4        3      4   2.0

In [8]:
# Target Distribution
target_df

,count,pct
heart_disease_diagnosis,,
1,149,74.5
0,51,25.5


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [12]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [13]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c7521-3300-74cc-b795-e1b028bbd79f
e95d6dc53b59612796fd457e7523c1717684755081f2f2db5f643d0ce44dc3f8
